# Benchmarking Forecasting Methods\n\nThis notebook compares how different forecasting methods affect the bullwhip effect\nunder a fixed Order-Up-To policy.\n\nWe compare:\n- **Naive**: Sample mean/std of full history\n- **MA(10)**: 10-period moving average\n- **MA(52)**: 52-period (1-year) moving average\n- **SES(0.3)**: Single exponential smoothing (alpha=0.3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from deepbullwhip.benchmark import BenchmarkRunner

runner = BenchmarkRunner(
    chain_config="semiconductor_4tier",
    demand="semiconductor_ar1",
    T=156,
    N=50,
    seed=42,
)

results = runner.run(
    policies=["order_up_to"],
    forecasters=[
        "naive",
        ("moving_average", {"window": 10}),
        ("moving_average", {"window": 52}),
        ("exponential_smoothing", {"alpha": 0.3}),
    ],
    metrics=["BWR", "FILL_RATE", "TC"],
)

pivot = results.pivot_table(
    index=["forecaster", "echelon"],
    columns="metric",
    values="value",
    aggfunc="mean",
)
print(pivot.to_string(float_format="%.3f"))

## BWR by Forecasting Method (Echelon 1)\n\nChen et al. (2000) showed that the MA window size directly affects the bullwhip lower bound.

In [ ]:
e1_bwr = results[(results["echelon"] == "E1") & (results["metric"] == "BWR")]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(e1_bwr["forecaster"], e1_bwr["value"], color=["#2196F3", "#4CAF50", "#FF9800", "#F44336"])
ax.set_xlabel("Bullwhip Ratio (BWR)")
ax.set_title("BWR at Echelon 1 by Forecasting Method")
ax.axvline(x=1.0, color="gray", linestyle="--", alpha=0.5, label="BWR=1 (no amplification)")
ax.legend()
plt.tight_layout()
plt.show()

## Chen Lower Bound Comparison

In [ ]:
from deepbullwhip.metrics.bounds import ChenLowerBound

bound = ChenLowerBound(lead_time=2, sensitivity=1.0, phi=0.72)
print(f"Chen et al. (2000) lower bound: BWR >= {bound.compute_bound():.3f}")
print(f"Observed BWR (Naive, E1): {e1_bwr[e1_bwr['forecaster']=='naive']['value'].values[0]:.3f}")